# NB03 - Manipulation et Transformations des données avec Databricks SQL

## Mise en place

Exécuter la cellules suivantes pour s'assurer créer / rafraichir les différents éléments du **Catalogue**

In [0]:
%run "./Resources/NB00/Setup"

Vérifier que l'on utilise le bon **Catalogue** et **Schéma** pour éviter de devoir renseigner le chemin complet dans les requêtes.

`SELECT current_catalog(), current_database();`

et pour changer les catalogues et schémas par défaut utiliser la requête : 

```sql
USE CATALOG labuser_<username>;
USE SCHEMA sql_analytics;
```

In [0]:
%sql
SELECT current_catalog(), current_database();

In [0]:
%sql
USE CATALOG labuser_edadou;
USE SCHEMA sql_analytics;

## Exploration

Pour explorer on peut soit utiliser le catalogue ou le faire par SQL comme décris dans les notebooks précédents.
Avec `SHOW TABLES` on peut récupérer la listes des tables présentes dans le schéma séléctionné et si on veut plus d'informations sur une table en particulier on peut exécuter la requête suivante : 

`DESCRIBE EXTENDED TABLE ca_orders;`

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
DESCRIBE EXTENDED ca_orders;

## Manipulation et transformations des données

1. Renommage de colonnes
2. Standardisation d'une colonne
3. Décoder une date en plusieurs éléments
4. Création de variable catégorique a partir des valeurs d'une colonne
5. JOIN
6. Création d'une nouvelle table **Silver**

In [0]:
%sql
SELECT * FROM ca_orders LIMIT 10;

### 1. Rennomage d'une colonne

En faisant le `SELECT` précédent on voit une colonne **salesrep_id** et on va renommer cette colonne en **sales_rep_id** :

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  sales_channel,
  order_status
FROM ca_orders
LIMIT 10;

### 2. Standardisation d'une colonne

Mettre les valeurs des colonnes **sales_channel** et **order_status** en majuscule :

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,   --renommage de la colonne
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  upper(sales_channel) as sales_channel,
  upper(order_status) as order_status
FROM ca_orders
LIMIT 10;

### 3. Décodage d'une date

Dans la colonne **order_date** on a une date au format *yyyy-MM-dd* et on va à partir de cette colonne généré 3 nouvelles colonnes : 
- order_year
- order_month
- order_day
- order_quarter

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  upper(sales_channel) as sales_channel,
  upper(order_status) as order_status,
  year(order_date) as order_year,
  month(order_date) as order_month,
  day(order_date) as order_day,
  quarter(order_date) as order_quarter
FROM ca_orders
LIMIT 10;

### 4. Création d'une colonne catégorique

On va créer une colonne **order_amount_size** qui va représenter la taille d'une commande.
On va dans un premier temps récupérer les terciles liés à la colonne **amount** avec : 

```sql
SELECT
    percentile_cont(0.3333) WITHIN GROUP (ORDER BY amount) AS seuil_petite_moyenne,
    percentile_cont(0.6667) WITHIN GROUP (ORDER BY amount) AS seuil_moyenne_grande
FROM ca_orders;
```

Pour définir les conditions suivantes : 
- Si < seuil_petite_moyenne = PETITE
- Si > seuil_petite_moyenne et < seuil_moyenne_grande = MOYENNE
- Sinon GRANDE

In [0]:
SELECT
    percentile_cont(0.3333) WITHIN GROUP (ORDER BY amount) AS seuil_petite_moyenne,
    percentile_cont(0.6667) WITHIN GROUP (ORDER BY amount) AS seuil_moyenne_grande
FROM ca_orders;

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  upper(sales_channel) as sales_channel,
  upper(order_status) as order_status,
  year(order_date) as order_year,
  month(order_date) as order_month,
  day(order_date) as order_day,
  quarter(order_date) as order_quarter,
  CASE
    WHEN amount < percentile_cont(0.3333) WITHIN GROUP (ORDER BY amount) OVER() THEN 'SMALL'
    WHEN amount < percentile_cont(0.6666) WITHIN GROUP (ORDER BY amount) OVER() THEN 'MEDIUM'
    ELSE 'LARGE'
  END as order_amount_size
FROM ca_orders
LIMIT 10;

### 5. JOIN

On va remplacer les **sales_rep_id** par le nom du représentant en faisant un **join** sur la table `ca_sales_reps`
et le **product_id** par **product_name** en faisant un **join** sur la table `ca_products`

In [0]:
select * from ca_sales_reps limit 10;

In [0]:
select * from ca_products limit 10;

In [0]:
SELECT
  o.order_line_id,
  o.order_id,
  o.order_date,
  o.customer_id,
  o.product_id,
  p.product_name as product_name,
  o.salesrep_id as sales_rep_id,
  r.salesrep_name as sales_rep_name,
  o.quantity,
  o.unit_price,
  o.discount_rate,
  o.amount,
  o.currency,
  upper(o.sales_channel) as sales_channel,
  upper(o.order_status) as order_status,
  year(o.order_date) as order_year,
  month(o.order_date) as order_month,
  day(o.order_date) as order_day,
  quarter(o.order_date) as order_quarter,
  CASE
    WHEN o.amount < percentile_cont(0.3333) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'SMALL'
    WHEN o.amount < percentile_cont(0.6666) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'MEDIUM'
    ELSE 'LARGE'
  END as order_amount_size
FROM ca_orders as o
INNER JOIN ca_sales_reps as r
ON o.salesrep_id = r.salesrep_id
INNER JOIN ca_products as p
ON o.product_id = p.product_id
LIMIT 10;

### 6. Création d'une table **Silver**

Maintenant que nous disposons d'une table contenant les formats, les colonnes et les jointures souhaités, nous allons créer la table **Silver** `ca_orders_silver`, qui contiendra les données nettoyées et enrichies.

In [0]:
create or replace table ca_orders_silver as
SELECT
  o.order_line_id,
  o.order_id,
  o.order_date,
  o.customer_id,
  o.product_id,
  p.product_name as product_name,
  o.salesrep_id as sales_rep_id,
  r.salesrep_name as sales_rep_name,
  o.quantity,
  o.unit_price,
  o.discount_rate,
  o.amount,
  o.currency,
  upper(o.sales_channel) as sales_channel,
  upper(o.order_status) as order_status,
  year(o.order_date) as order_year,
  month(o.order_date) as order_month,
  day(o.order_date) as order_day,
  quarter(o.order_date) as order_quarter,
  CASE
    WHEN o.amount < percentile_cont(0.3333) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'SMALL'
    WHEN o.amount < percentile_cont(0.6666) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'MEDIUM'
    ELSE 'LARGE'
  END as order_amount_size
FROM ca_orders as o
INNER JOIN ca_sales_reps as r
ON o.salesrep_id = r.salesrep_id
INNER JOIN ca_products as p
ON o.product_id = p.product_id;

In [0]:
select * from ca_orders_silver;

In [0]:
DESCRIBE EXTENDED ca_orders_silver;

Si on va dans le catalogue, peut aller voir les détails de la nouvelle table et comment elle est créée / alimentée avec son **lineage** :

![image_1790869482980.png](./Resources/image_1790869482980.png "image_1790869482980.png")

## Analyse

Maintenant qu'on a une table enrichie avec toutes les données souhaitées on peut faire de l'analyse dessus.

1. Filtres

On va récupérer que les commandes de *Chloe Durand* triées par **amount**

In [0]:
select distinct(sales_rep_name) from ca_orders_silver;